---
<font color='Blue' size="4">
F37.206 컴퓨팅 탐색: 실생활에서 활용하기(Exploring Computing: Applications in Everyday Life)</font>

---


# Chapter 13. 동적 웹 정보수집 자동화

<div style="background-color: #f5fff5; padding: 10px; border-radius: 5px; color: #000000;">

<font size=5> <strong> <mark style="background-color: #f5fff5;">✅ 학습목표와 기대효과 </mark></strong></font>

<mark style="background-color: #f5fff5;">
🔹 학습목표<br>
  <ul><li> selenium 모듈에 대해서 알아보자.</li>
  <li> 웹드라이버를 활용하여 웹페이지를 크롤링 해보자.</li>
  <li> 크롤링을 지원하는 중요한 메서드들을 알아보자.</li></ul> 
🔹 기대효과<br>
  <ul><li> 웹 크롤링에 필요한 핵심 모듈과 메서드를 이해하여 원하는 정보를 자동으로 수집할 수 있다.</li>
  <li>실습을 통해 웹데이터를 활용한 자동화와 데이터 분석의 기초 역량을 향상시킨다.</li></ul></mark>
</div>

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**셀레니움이란?**</mark></div>

- 요즘 대부분의 웹페이지는 HTML안에 자바스크립트가 들어가 있어서 동적으로 사용자와 소통한다. 즉, 웹페이지가 사용자의 키보드 입력, 사용자의 마우스 클릭에 동적으로 반응한다는 의미이다.
- 이와 같이 자바스크립트를 활용한 웹 페이지를 크롤링 하려면 자바스크립트를 해석할 수 있는 크롤러가 필요하다.
- 셀레니움은 자바스크립트가 적용된 웹페이지에서 스크랩핑을 지원하는 모듈로 웹드라이버 기능을 활용하여 다양한 웹브라우저를 자동으로 동작시킨다.

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**설치 및 환경설정**</mark></div>

- 셀레니움을 동작시키기 위해서는 한글폰트, 셀레니움, 크롬 드라이버 등의 설치가 필요하다.

### 웹브라우저 한글폰트 깨짐 방지
- 웹브라우저를 구동시켜 웹페이지에 접속했을 때 한글 폰트가 깨져서 네모로 나오는 것을 방지한다.
- `%matplotlib inline`은 Matplotlib로 그린 그래프를 .ipynb(노트북파일) 안에서 바로 보고자 할때 넣어준다.

In [ ]:
%matplotlib inline  
import matplotlib.pyplot as plt
from matplotlib import font_manager
import platform

if platform.system() == 'Darwin':  # Mac OS
    # 시스템 폰트 확인
    for font in font_manager.findSystemFonts():
        if "Nanum" in font or "AppleGothic" in font:
            print(font)
    # 폰트 지정 (예: AppleGothic)
    plt.rc('font', family='AppleGothic')
else:  # Windows
    font_path = "C:/Windows/Fonts/NanumGothic.ttf"
    font_name = font_manager.FontProperties(fname=font_path).get_name()
    plt.rc('font', family=font_name)

# 테스트 그래프
plt.plot([1,2,3],[1,4,9])
plt.title("테스트 그래프")
plt.show()

### 셀레니움 설치
- 동적 웹크롤링/스크랩핑을 위한 모듈인 셀레니움을 설치한다.

In [ ]:
pip install selenium

### 크롬 드라이버 자동 설치 모듈 설치
- 크롬 드라이버 자동 설치에 필요한 모듈을 설치한다.

In [3]:
pip install chromedriver-autoinstaller

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0 -> 24.0
[notice] To update, run: d:\users\anaconda3\python.exe -m pip install --upgrade pip


### 크롬드라이버 설치 및 테스트
- 적절한 버전의 크롬드라이버(chromedriver.exe)를 자동 설치 및 실행한다.
- 크롬 브라우저가 잘 동작하는지 테스트해보기 위해서 구글 홈페이지에 접속해본다.

In [4]:
from selenium import webdriver
import chromedriver_autoinstaller

# 크롬드라이버 자동 설치 및 실행
chromedriver_autoinstaller.install()
driver = webdriver.Chrome()

driver.get("https://www.google.com")
print(driver.title)

Google


## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**네이버 웹크롤링**</mark></div>

### 1. 네이버 접속하기
- 이제 웹드라이버를 통해 네이버에 접속해보자.
- 웹드라이버를 통해 웹페이지에 접속할 때에는 `driver.get(url)` 함수를 사용한다.
- 웹페이지에 접속은 네트워크 상황에 따라 몇 초 정도 소요될 수도 있다. 따라서 로딩되는 시간을 기다려야 한다.
- `driver.implicitly_wait(s)`은 웹드라이버에게 웹페이지가 로딩되는 시간 또는 엘리먼트를 찾는데 걸리는 최대 대기 시간을 설정하는 함수이다.

:::{admonition} implicitly_wait() vs. time.sleep()
:class: tip  

- implicitly_wait(s):
    - 웹드라이버에게 최대 s초 동안 대기하도록 하는 메서드이다. 
    - 웹페이지가 로딩되는 시간 또는 엘리먼트를 찾는데 걸리는 최대 대기 시간을 설정하는 메서드이다. 
    - s초가 지나도 작업을 완료하지 못하면 NoSuchElementException 에러를 발생시킨다. 즉, 웹페이지가 준비도 안되었는데 엘리먼트를 찾으라 하면 NoSuchElementException 에러가 발생한다.

- time.sleep(s): 
    - time 모듈에 포함된 함수로 실행을 s초 동안 일시 중지하였다가 시간이 경과하면 실행이 계속된다.
    - time.sleep(s)은 웹페이지가 로딩되는 시간을 기다리는데 유용하게 사용할 수 있다.
    - 이 경우, time.sleep(5)를 호출하면 프로그램 실행이 5초 동안 일시 중지된다. 

​이 두 코드는 서로 목적이 다르지만, 웹 스크래핑 작업에서 적절한 시점에 기다리는 용도로 사용할 수 있다.

:::

- `driver.title`는 웹페이지 잘 접속 되었는지 확인하기 위해 웹페이지의 제목표시줄의 내용을 출력한다.
- `driver.save_screenshot('파일명')`은 웹페이지에 잘 접속 되었는지 확인하기 위해 웹페이지의 스크린샷을 찍는다. 스크린샷이 성공적으로 찍히면 True를 반환한다. VS code에서는 크롬 브라우저가 동작하는 모습을 눈으로 확인할 수 있기 때문에 스크린샷을 찍지 않아도 된다. 

In [ ]:
url = "http://naver.com"
driver.get(url)
driver.implicitly_wait(3)
print(driver.title)
#driver.save_screenshot('screenshot.png')

- 동적 웹페이지에서 웹크롤링을 하려면 **사람이 하는 동작 그대로 코드에게 일을 시켜야 한다.**
- 네이버 검색어 입력창에 검색어가 자동으로 입력되도록 해보자. 사람은 눈으로 검색창을 쉽게 찾을 수 있지만, 코드에서는 검색창의 위치를 명시적으로 찾아야 한다. 
- 즉, 검색어 입력박스의 엘리먼트(element)를 찾아서 알려줘야 한다.
- 엘리먼트(Element) 는 웹페이지 안의 하나의 HTML 구성요소(태그) 를 의미한다. 웹페이지를 구성하는 버튼, 입력창, 링크, 이미지, 텍스트 박스 같은 모든 개별 객체가 다 엘리먼트이다.

<div align="center"><img src="https://haesunbyun.github.io/common/images/selenium.png" style="width:700px;"></div>

### 2. 태그와 속성 확인
- 개발자 도구를 활용하여 네이버 검색어 입력박스 엘리먼트의 태그와 속성을 확인한다.
- 태그 구조는 다음과 같다.

  ```html
  <태그 속성명="속성값" 속성명="속성값" ...>컨텐츠</태그>
- 검색어 입력박스의 태그와 속성은 `<input id="query" name="query" ...>`이다. 기본적으로 입력박스의 태그는 <input>이다.

### 3. 코드로 엘리먼트 검색
- 검색어 입력박스의 태그와 속성을 확인했으니 이제 코드로 그 엘리먼트를 찾도록 해야 한다.
- 셀레니움에서 엘리먼트를 찾는 방법은 다음과 같다.
  - find_element(): 하나의 엘리먼트를 찾는다.
  - find_elements(): 여러개의 엘리먼트를 찾는다.

- find_element(), find_elements() 함수의 괄호안에는 추출할 옵션을 넣어준다. 옵션의 종류는 다음과 같다.

  | 옵션 | 설명 |
  |:--------|:------|
  | `By.ID` | 태그의 `id` 값으로 추출 |
  | `By.NAME` | 태그의 `name` 값으로 추출 |
  | `By.XPATH` | 태그의 경로(`XPath`)로 추출 |
  | `By.LINK_TEXT` | 링크의 전체 텍스트 값으로 추출 |
  | `By.PARTIAL_LINK_TEXT` | 링크 텍스트의 일부(자식 텍스트) 값으로 추출 |
  | `By.TAG_NAME` | 태그명으로 추출 |
  | `By.CLASS_NAME` | 태그의 클래스명으로 추출 |
  | `By.CSS_SELECTOR` | CSS 선택자로 추출 |


- 위 옵션을 사용하려면 By 클래스를 먼저 import해야 한다.
  - <font color="red">`from selenium.webdriver.common.by import By`</font>

- 예를 들어, 검색어 입력박스의 속성가운데 하나인 ID로 찾는다면 find_element(By.ID, "query")로 넣어준다.
- 검색어 입력박스 엘리먼트를 찾았다면 `엘리먼트객체.send_keys()` 괄호안에 검색어를 넣어준다.
- 아래 코드에서는 엘리먼트를 box_elem으로 저장했으므로 box_elem.send_keys('공휴일')과 같이 작성하였다.
- 검색박스의 문자열을 지우려면 엘리먼트.clear()하면 된다. 지금 당장은 지우지 말자.

In [6]:
from selenium.webdriver.common.by import By
box_elem = driver.find_element(By.ID, "query") #검색어 입력박스
box_elem.send_keys('공휴일') #검색어 입력

### 4. 검색버튼 클릭되게 하기
- 검색어가 입력된 것을 확인했다면 검색버튼이 클릭되게 해보자.
- 이때 버튼과 같이 클릭되는 항목들은 일반적으로 개발자도구에서 xpath를 찾아 넣어주면 편하다.
- xpath는 XML Path Language로, 웹페이지의 특정 엘리먼트나 속성에 접근하기 위한 경로이다.

- 검색버튼의 xpath를 찾기 위해 개발자 도구를 활용해보자.
- 개발자도구 > 엘리먼트 선택 버튼 > 네이버의 검색버튼으로 가서 클릭해보자. 오른쪽 개발자도구의 소스코드영역에 해당 검색버튼이 있는 html 코드가 있을 것이다.
- 그 html 코드 위에서 마우스 오른쪽 버튼을 클릭하여 copy > copy xpath를 클릭한다.
- 그러면 xpath 값이 클립보드에 저장된다. 이를 변수 xpath에 붙여넣기 하여 문자열로 저장한다.
- find_element(By.XPATH, value=xpath)로 검색버튼 엘리먼트를 찾아 click()함수로 클릭되게 한다.
- 검색 버튼이 클릭되면 검색 결과 페이지가 나온다.
- 이전 페이지로 이동하려면 driver.back()한다.

In [ ]:
xpath = '여러분이 채워주세요.'    #검색버튼 xpath
driver.find_element(By.XPATH, value=xpath).click()
driver.implicitly_wait(3)

### 🚀**해보기 1: 검색어 입력받아 검색하기**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
검색어를 input() 함수로 입력받아 selenium으로 네이버에서 검색하도록 소스코드를 작성해보세요. 
</mark></div>

### 🚀**해보기 2: 네이버 블로그 검색하기**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
아래 네이버 블로그 사이트에서 selenium으로 '겨울온천'을 검색하도록 코드를 작성하시오.

https://section.blog.naver.com/
</mark></div>

### 5. 웹페이지 컨텐츠 추출하기
- 이제 웹페이지 검색결과에서 필요한 데이터를 추출해보자.
- 이전에 배웠던 BeautifulSoup과 selenium 중 어느것을 사용해도 좋다.


#### 5.1 BeautifulSoup으로 추출하기
- 셀레니움으로 접근한 웹페이지를 가져올때에는 `driver.page_source`를 통해 HTML 코드를 가져온다.
- 이후 `BeautifulSoup(html, 'html.parser')`으로 파싱한다.
- 추출하려고 하는 컨텐츠가 어떤 태그와 속성을 가지고 있는지 개발자 도구를 활용하여 알아낸 후, find_all()로 검색한다.
- 검색된 결과에서 필요한 컨텐츠만 추출한다. 아래 예제에서는 검색된 목록의 제목을 추출해보자.

In [ ]:
from bs4 import BeautifulSoup

html=driver.page_source 
soup = BeautifulSoup(html, 'html.parser')
subject_list = soup.find_all("채워주세요.")
# 작성해주세요.

#### 5.2 셀레니움으로 추출하기
- 셀레니움으로 추출하면 페이지를 가져올 필요없이 `driver.find_elements()` 직접 추출이 가능하다.
- 단, 셀리니움으로 찾은 결과는 BeautifulSoup에서와 달리 selenium.webdriver.remote.webelement.WebElement로 나온다. 
- 아래 코드에서 print(subject_list)를 해보고 그 결과를 확인하자.

In [ ]:
subject_list = driver.find_elements(By.CLASS_NAME, "title_post" )
subject_list

In [ ]:
for each in subject_list:
  print(each.text)

### 6. 페이지 이동하면서 컨텐츠 추출하기

- 1페이지의 제목은 추출했으니 2페이지, 3페이지 이동하면서 제목을 추출해보자.
- 사람이 2페이지 검색결과를 보려면 페이지번호 중에 2를 클릭 할 것이다. 이 클릭을 나 대신 코드가 클릭하도록 해줘야 한다.
- 이를 위해 개발자 도구를 활용하여 페이지번호 2의 xpath를 추출한다.
- xpath를 활용하여 그 엘리먼트가 클릭되게 한 후, 페이지가 로딩되기를 기다리자. 
- 로딩되었다면 해당 페이지의 제목을 추출해보자.

In [ ]:
xpath = '채워주세요.'
driver.find_element(By.XPATH, value=xpath).click()
driver.implicitly_wait(10)
subject_list = driver.find_elements(By.CLASS_NAME, "title_post" )
for each in subject_list:
  print(each.text)

### 7. 반복적으로 일시키기
- 6번에서 했던 절차를 반복하면 된다.
- 개발자 도구를 활용하여 페이지번호 2, 페이지번호 3, 페이지번호 4 등의 xpath를 확인해 보고 어느 부분이 바뀌고 있는지 체크하자.
- xpath의 값이 거의 비슷할 것인데 span[2], span[3], span[4]와 같이 페이지번호에 따라 span[] 대괄호안의 숫자만 달라지고 있는 것을 볼 수 있을 것이다.
- 반복문으로 실행시키기 딱 좋은 구조이다. 2, 3, 4라는 상수값을 변수로 줘서 반복문으로 실행시킬 수 있다.
- 1페이지부터 5페이지까지 제목만 추출해서 리스트로 만들어보자.

In [ ]:
import time
titleList=[]
for i in range(1,6):
  xpath = f'채워주세요.'
  driver.find_element(by=By.XPATH, value=xpath).click()

  time.sleep(3)
  subject_list = driver.find_elements(By.CLASS_NAME, "title_post" )

  for each in subject_list:
    titleList.append(each.text.strip())
  print(f'-------{i}/5')

print(titleList)

### 8. 속성값 추출하기

- 블로그에서 검색어에 대한 제목뿐만 아니라 블로그 링크까지 추출해보자.
- 블로그 링크의 주소는 일반적으로 <a> 태그에 있으나, 단순히 <a> 태그만 검색하면 의도하지 않은 다른 링크들까지 모두 포함될 수 있다.
- 따라서 검색 범위를 더 구체적으로 지정하자. <a> 태그와 필요한 속성값을 함께 조건으로 사용하면 검색 범위를 좁힐 수 있다. 이때 속성명으로는 class가 많이 사용된다.
- 찾은 엘리먼트에서 특정 속성값을 가져올 때는 `get_attribute()` 메서드의 괄호 안에 속성명을 넣어 호출하면 된다.

In [ ]:
a_tag_list = driver.find_elements(By.CLASS_NAME, 'desc_inner')
a_tag_list[0].get_attribute('ng-href')

- 반복문으로 제목과 링크를 모두 딕셔너리 리스트로 만들어보자.
    ```python
    dict_lists = [
        {"title": "제목1", "url": "https://news.naver.com"},
        {"title": "제목2", "url": "https://google.com"}
    ]
    ```

In [ ]:
import time
dict_lists = list()

for i in range(1,6):
  xpath = f'//*[@id="content"]/section/div[3]/span[{i}]/a'
  driver.find_element(by=By.XPATH, value=xpath).click()

  time.sleep(3)
  subject_list = driver.find_elements(By.CLASS_NAME, "title_post" )
  a_tag_list = driver.find_elements(By.CLASS_NAME, 'desc_inner')

  for sub, link in zip(subject_list, a_tag_list):
    link_dict=dict()
    link_dict['title'] = sub.text.strip()
    link_dict['url'] = link.get_attribute('ng-href')
    dict_lists.append(link_dict)
  print(f'-------{i}/5')

print(dict_lists)

### 9. 제어 종료
- driver.close()를 통해 웹드라이버를 제어 종료한다.

In [7]:
driver.quit() # 크롬 드라이버를 종료할 때 실행

- 지금까지 셀레니움을 사용해 웹 데이터를 추출하는 방법을 익혀보았다.
- 하지만 아쉽게도 Streamlit 환경에서는 Selenium이 원활하게 동작하지 않는다.
- 이는 Streamlit이 웹 기반 앱으로 실행되며, 서버와 사용자의 브라우저가 분리된 구조를 갖고 있기 때문이다.
- Selenium은 ChromeDriver(또는 Edge 등)를 실행해 로컬 PC에서 실제 브라우저 창을 띄우는 방식으로 작동한다.
- 반면 Streamlit은 서버에서 실행되고, 사용자는 웹 브라우저를 통해 화면만 확인하기 때문에 서버에서 띄운 브라우저 창이 사용자에게 보이지 않을 뿐 아니라, 환경에 따라 브라우저 자체를 실행할 수 없는 경우도 있다.
- 그렇다면 어떻게 해야 할까? 셀레니움으로 데이터를 수집해 파일로 저장한 뒤, Streamlit에서 그 파일을 불러와 시각화하거나 출력하는 방식으로 해결할 수 있다.

### 🚀**해보기 3: 네이버의 쇼핑 검색하기**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
검색어를 입력받아 아래 네이버 쇼핑탭에서 selenium으로 검색어를 검색하고, 1페이지에 검색된 상품 가운데 최고가, 최저가를 출력하시오.

https://shopping.naver.com/home
</mark></div>

네이버의 쇼핑페이지는 마우스로 페이지를 스크롤링하거나 스크롤바를 내리면 상품 목록이 나오도록 되어 있습니다. 스크롤링하는 코드는 다음과 같습니다.
아래 코드는 윈도우의 최상단으로부터 끝(document.body.scrollHeight)까지 내리는 코드입니다. 숫자로 조정 할 수도 있습니다. (예:3000)

```python
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(3)
```

Sample
```python
검색상품명: 건조기
48개 검색되었습니다.
최고가: 1656060원
최저가: 29900원
```


## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**마무리**</mark></div>
오늘은 셀레니움을 활용해 웹에서 필요한 정보를 자동으로 추출하는 방법을 배워보았다. 셀레니움은 실제 브라우저를 제어하며 데이터를 수집할 수 있다는 점에서 매우 강력한 도구이다. 앞으로 더 복잡한 페이지 구조나 동적 데이터도 충분히 다룰 수 있을 것이며, 이를 기반으로 다양한 프로젝트에 응용해볼 수 있을 것이다. 오늘 배운 내용을 잘 정리해두면 이후 데이터 자동화나 크롤링 작업에서 큰 도움이 될 것이다.

---
<font color='Grey' size="4">
F37.206 컴퓨팅 탐색: 실생활에서 활용하기(Exploring Computing: Applications in Everyday Life)</font>

---
서울대학교 학부대학 강의교수 변해선